# Intro figure: deoxygenation by depth range

This plotting-only notebook uses the cached statistics produced by the source analysis in `../Paper_figures`. It creates a single-row, three-panel intro figure and saves **only** the PDF version in `Figures/`.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

SOURCE_DIR = Path('../Paper_figures')
CACHE_DIR = SOURCE_DIR / 'cache_three_panel_o2_potden_en4_1965_2021'
OUTPUT_FILE = Path('Figures/Intro_figure_deoxygenation_three_depths.pdf')
COVERAGE_CACHE=CACHE_DIR/'annual_doxy_depth_layer_counts_1965_2021_fan_all_depth.csv'
YEAR_START,YEAR_END=1965,2021
BASELINE_START,BASELINE_END=2010,2019

DEPTH_LAYERS = {
    'full_depth': {
        'label': 'Full water column',
        'cache': CACHE_DIR / 'full_depth_plot_data_1965_2021_6_products_olivelli_potden_en4_v2.pkl',
    },
    'upper_2000m': {
        'label': 'Above 2000 m',
        'cache': CACHE_DIR / 'upper_2000m_plot_data_1965_2021_6_products_olivelli_potden_en4_v2.pkl',
    },
    'below_2000m': {
        'label': 'Below 2000 m',
        'cache': CACHE_DIR / 'below_2000m_plot_data_1965_2021_6_products_olivelli_potden_en4_v2.pkl',
    },
}

# Olivelli et al. (2026) is intentionally purple so it is distinct from
# Ito et al. (2024): Neural network, which is green.
MAPPING_CONFIG = {
    'Ito (2022)': {'color': '#0072B2', 'marker': 's'},
    'Ito et al. (2024): Neural network': {'color': '#009E73', 'marker': '^'},
    'Ito et al. (2024): Random forest': {'color': '#D55E00', 'marker': 'o'},
    'Gouretski et al. (2024)': {'color': '#CC79A7', 'marker': 'D'},
    'Roach & Bindoff (2023)': {'color': '#E69F00', 'marker': 'P'},
    'Olivelli et al. (2026)': {'color': '#7B2CBF', 'marker': 'X'},
}

GROUP_LABELS = [
    'Mapped\nobservational\nproduct fields',
    'Fully sampled\nmodel fields',
    'ESM sampled-\nand-mapped\nfields',
    'EC-inferred\nfully-sampled\nproduct fields',
]
Y_POSITIONS = [3, 2, 1, 0]

layer_results = {}
for key, cfg in DEPTH_LAYERS.items():
    if not cfg['cache'].exists():
        raise FileNotFoundError(f'Missing cache: {cfg["cache"].resolve()}')
    with cfg['cache'].open('rb') as handle:
        layer_results[key] = pickle.load(handle)

depth_layer_counts=pd.read_csv(COVERAGE_CACHE,index_col='year').reindex(range(YEAR_START,YEAR_END+1),fill_value=0).astype(int)

In [ ]:
def product_display_label(name):
    if name == 'Ito (2022)':
        return r'Ito (2022)$^{\dagger}$'
    if name.startswith('Ito et al. (2024)'):
        return name.replace('Ito et al. (2024)', r'Ito et al. (2024)$^{*}$')
    if name == 'Roach & Bindoff (2023)':
        return r'Roach & Bindoff (2023)$^{\ddagger}$'
    if name == 'Olivelli et al. (2026)':
        return r'Olivelli et al. (2026)$^{\P}$'
    return name


def anomaly_series(s):
    b=s.loc[BASELINE_START:BASELINE_END]
    return s-(b.mean() if len(b) else s.iloc[:10].mean())


def finite_values(values):
    values = np.asarray(list(values), dtype=float)
    return values[np.isfinite(values)]


def retained_ec_items(ec):
    return [(name, result) for name, result in ec.items()
            if result and result.get('significant_positive_slope', False)]


def pdf_quantile(x, pdf, probability):
    x = np.asarray(x, dtype=float)
    pdf = np.asarray(pdf, dtype=float)
    increments = 0.5 * (pdf[1:] + pdf[:-1]) * np.diff(x)
    cdf = np.concatenate(([0.0], np.cumsum(increments)))
    if not len(cdf) or not np.isfinite(cdf[-1]) or cdf[-1] <= 0:
        return np.nan
    cdf /= cdf[-1]
    return float(np.interp(probability, cdf, x))


def pdf_box_stats(mixture):
    x = np.asarray(mixture['xgrid'], dtype=float)
    pdf = np.asarray(mixture['pdf'], dtype=float)
    if not len(x) or not np.any(np.isfinite(pdf)) or np.nanmax(pdf) <= 0:
        return None
    return {
        'med': pdf_quantile(x, pdf, 0.50),
        'q1': pdf_quantile(x, pdf, 0.25),
        'q3': pdf_quantile(x, pdf, 0.75),
        'whislo': mixture['lower'],
        'whishi': mixture['upper'],
        'fliers': [],
    }


def panel_limit_values(result):
    values = [0.0]
    values.extend(finite_values(result['full_trend'].values()))
    values.extend(finite_values(result['obs_trend'].values()))
    values.extend(finite_values(v for d in result['gap_trend'].values() for v in d.values()))
    for _, ec_result in retained_ec_items(result['ec_trend']):
        values.extend(finite_values([ec_result['x_lo'], ec_result['x_ec'], ec_result['x_hi']]))
    mixture = result['trend_mixture']
    values.extend(finite_values([mixture['lower'], mixture['median'], mixture['upper']]))
    return np.asarray(values, dtype=float)


def draw_panel(ax, result, title, show_group_labels=False):
    full = finite_values(result['full_trend'].values())
    observed = finite_values(result['obs_trend'].values())
    gap = finite_values(v for d in result['gap_trend'].values() for v in d.values())

    ax.boxplot(
        [observed, full, gap], vert=False, positions=Y_POSITIONS[:3], widths=0.48,
        patch_artist=True, showfliers=False,
        boxprops=dict(facecolor='0.90', edgecolor='0.25', linewidth=1.1),
        medianprops=dict(color='k', linewidth=1.5),
        whiskerprops=dict(color='0.25', linewidth=1.1),
        capprops=dict(color='0.25', linewidth=1.1),
    )
    ec_stats = pdf_box_stats(result['trend_mixture'])
    if ec_stats is not None:
        ax.bxp(
            [ec_stats], positions=[0], vert=False, widths=0.48, patch_artist=True,
            showfliers=False,
            boxprops=dict(facecolor='0.90', edgecolor='0.25', linewidth=1.1),
            medianprops=dict(color='k', linewidth=1.5),
            whiskerprops=dict(color='0.25', linewidth=1.1),
            capprops=dict(color='0.25', linewidth=1.1),
        )

    rng = np.random.default_rng(42)
    ax.scatter(full, 2 + rng.uniform(-0.10, 0.10, len(full)), s=48, marker='*',
               color='0.45', alpha=0.8, zorder=3)
    for name, by_model in result['gap_trend'].items():
        values = finite_values(by_model.values())
        cfg = MAPPING_CONFIG[name]
        ax.scatter(values, 1 + rng.uniform(-0.13, 0.13, len(values)), s=24,
                   marker=cfg['marker'], facecolor='0.55', edgecolor='0.30',
                   linewidth=0.4, alpha=0.65, zorder=3)
    for name, value in result['obs_trend'].items():
        if np.isfinite(value):
            cfg = MAPPING_CONFIG[name]
            ax.scatter(value, 3, s=72, color=cfg['color'], marker=cfg['marker'],
                       edgecolor='k', linewidth=0.4, zorder=5)
    for name, ec_result in retained_ec_items(result['ec_trend']):
        cfg = MAPPING_CONFIG[name]
        ax.scatter(ec_result['x_ec'], 0, s=72, color=cfg['color'], marker=cfg['marker'],
                   edgecolor='k', linewidth=0.4, zorder=5)

    ax.axvline(0, color='0.25', lw=0.9, ls='--', zorder=0)
    ax.set_title(title, fontweight='bold')
    ax.set_yticks(Y_POSITIONS)
    ax.set_yticklabels(GROUP_LABELS if show_group_labels else [])
    ax.set_ylim(-0.55, 3.55)
    ax.grid(True, axis='x', alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

In [ ]:
plt.rcParams.update({'font.size':10,'axes.labelsize':11,'axes.titlesize':12,'legend.fontsize':10})
fig=plt.figure(figsize=(14,10.5));g=fig.add_gridspec(3,3,height_ratios=[1.15,.38,1],hspace=.42,wspace=.1)
a=fig.add_subplot(g[0,:]);b=fig.add_subplot(g[1,:],sharex=a);axs=[fig.add_subplot(g[2,i]) for i in range(3)]
R=layer_results['full_depth']
for s in R['model_ts'].values():
 z=anomaly_series(s);a.plot(z.index,z,color='.68',lw=1,alpha=.75)
d=pd.concat({k:anomaly_series(v) for k,v in R['model_ts'].items()},axis=1);a.plot(d.index,d.mean(axis=1),color='k',lw=2.4)
for name,s in R['obs_ts'].items():
 c=MAPPING_CONFIG[name];z=anomaly_series(s);a.plot(z.index,z,color=c['color'],marker=c['marker'],markevery=max(1,len(z)//12),ms=(7 if name == 'Olivelli et al. (2026)' else 4),lw=2,markeredgecolor='k',markeredgewidth=.4)
a.axhline(0,color='.2',lw=.8);a.set(xlim=(YEAR_START,YEAR_END),ylim=(-1.5,None),ylabel=r'Global O$_2$ anomaly [$\mu$mol kg$^{-1}$]',title='a.  Full-water-column oxygen anomalies');a.tick_params(labelbottom=False);a.grid(alpha=.25);a.title.set_fontweight('bold')
H=[Line2D([0],[0],color='.68',lw=1.5,label='Individual CMIP6 models'),Line2D([0],[0],color='k',lw=2.4,label='CMIP6 mean')]+[Line2D([0],[0],color=c['color'],marker=c['marker'],lw=2,label=product_display_label(n)) for n,c in MAPPING_CONFIG.items()];a.legend(handles=H,frameon=False,ncol=4,loc='lower left')
C={'osd':'#56B4E9','ctd':'#777777','Argo':'#E69F00','unknown':'#BBBBBB'};L={'osd':'OSD','ctd':'CTD','Argo':'Argo','unknown':'Unknown'};y=depth_layer_counts.index.to_numpy();bot=np.zeros(len(y));src=[]
for col in depth_layer_counts:
 s,_=col.split('|',1)
 if s not in src:src.append(s)
for s in src:
 u=depth_layer_counts.get(f'{s}|Upper 2000 m',pd.Series(0,index=depth_layer_counts.index)).to_numpy(float);d=depth_layer_counts.get(f'{s}|Below 2000 m',pd.Series(0,index=depth_layer_counts.index)).to_numpy(float);c=C.get(s,'.6');b.bar(y,u,bottom=bot,width=.86,color=c,edgecolor='white',linewidth=.2);b.bar(y,d,bottom=bot+u,width=.86,color=c,edgecolor='.2',linewidth=.25,hatch='///');bot+=u+d
b.set_yscale('log');b.set(ylabel='Number of O$_2$\nmeasurements',xlabel='Year',title='b.  Observational coverage');b.title.set_fontweight('bold');b.grid(axis='y',alpha=.25);b.spines[['top','right']].set_visible(False)
H=[Patch(facecolor=C.get(s,'.6'),edgecolor='none',label=L.get(s,s)) for s in src]+[Patch(facecolor='white',edgecolor='.25',label='Above 2000 m'),Patch(facecolor='white',edgecolor='.25',hatch='///',label='Below 2000 m')];b.legend(handles=H,frameon=False,ncol=6,loc='upper left')
v=np.concatenate([panel_limit_values(r) for r in layer_results.values()]);lo,hi=np.nanmin(v),np.nanmax(v);sp=hi-lo
for i,((k,c),ax) in enumerate(zip(DEPTH_LAYERS.items(),axs)):
 draw_panel(ax,layer_results[k],f'{chr(99+i)}.  {c["label"]}',show_group_labels=i==0);ax.set_xlim(lo-.06*sp,hi+.06*sp);ax.set_xlabel(r'Deoxygenation [$\mu$mol kg$^{-1}$ decade$^{-1}$]')
H=[Line2D([0],[0],color='none',marker=c['marker'],markerfacecolor=c['color'],markeredgecolor='k',markersize=7,label=product_display_label(n)) for n,c in MAPPING_CONFIG.items()];fig.legend(handles=H,loc='lower center',bbox_to_anchor=(.5,.005),ncol=6,frameon=False);fig.subplots_adjust(left=.12,right=.985,bottom=.13,top=.965);OUTPUT_FILE.parent.mkdir(exist_ok=True);fig.savefig(OUTPUT_FILE,bbox_inches='tight',facecolor='white');plt.show()
